# Final Evaluation and Model Comparison

**Notebook 10 of 10** | LLM Alignment Series

This notebook provides a comprehensive side-by-side evaluation of every model produced during the alignment series. We load each model sequentially (to stay within 24 GB VRAM), generate responses on a shared set of test prompts, and compare across multiple dimensions:

| Model | Source | Method |
|-------|--------|--------|
| **Base** | Qwen2.5-7B-Instruct | Pre-trained (no fine-tuning) |
| **SFT** | Notebook 03 | Supervised fine-tuning on OASST1 |
| **RLHF-GRPO** | Notebook 05 | GRPO with trained reward model |
| **DPO** | Notebook 06 | Direct preference optimisation on UltraFeedback |
| **GRPO-Custom** | Notebook 08 | GRPO with rule-based reward functions |
| **f-GRPO (KL)** | Notebook 09 | f-GRPO with KL divergence |
| **f-GRPO (RevKL)** | Notebook 09 | f-GRPO with Reverse KL divergence |
| **f-GRPO (Hellinger)** | Notebook 09 | f-GRPO with Hellinger divergence |

### Evaluation Dimensions

1. **Reward scores** — format, conciseness, non-repetition, helpfulness
2. **Response characteristics** — length distribution, vocabulary diversity
3. **Qualitative comparison** — side-by-side responses on diverse prompts
4. **Head-to-head win rates** — pairwise comparison matrix
5. **Aggregate ranking** — overall model ranking

**Model**: Qwen2.5-7B-Instruct | **GPU**: NVIDIA RTX 4090

## Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import re
import gc
import os
import json
import textwrap
from collections import defaultdict

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")

## Test Prompts

We use a diverse set of prompts spanning five categories to stress-test different model capabilities.

In [ ]:
TEST_PROMPTS = {
    "factual": [
        "What causes earthquakes?",
        "Explain how vaccines work.",
        "What is the greenhouse effect?",
        "How does the internet work?",
    ],
    "reasoning": [
        "What are the pros and cons of remote work?",
        "Compare renewable and non-renewable energy sources.",
        "What are the ethical implications of genetic engineering?",
        "What factors contribute to income inequality?",
    ],
    "instruction": [
        "List 5 tips for better sleep. Use a numbered list.",
        "Give me a step-by-step guide to making scrambled eggs.",
        "Explain the scientific method in exactly 4 steps.",
        "List 3 ways to reduce your carbon footprint.",
    ],
    "creative": [
        "Write a haiku about the ocean.",
        "Describe a futuristic city in 3 sentences.",
        "Write a motivational message for someone starting a new job.",
        "Create a short analogy explaining how memory works.",
    ],
    "advice": [
        "What advice would you give someone learning to code?",
        "How can someone manage stress effectively?",
        "What should someone consider before adopting a pet?",
        "How can a student improve their study habits?",
    ],
}

# Flatten for generation
all_prompts = []
prompt_categories = []
for cat, prompts in TEST_PROMPTS.items():
    for p in prompts:
        all_prompts.append(p)
        prompt_categories.append(cat)

print(f"Total test prompts: {len(all_prompts)}")
for cat, prompts in TEST_PROMPTS.items():
    print(f"  {cat}: {len(prompts)}")

## Reward Functions

Identical to Notebooks 08 and 09 — four complementary reward signals.

In [ ]:
def format_reward(text: str) -> float:
    """Reward structured formatting."""
    score = 0.0
    if re.search(r'\d+\.\s', text):
        score += 0.5
    if re.search(r'[-*]\s', text):
        score += 0.3
    if text.count('\n\n') >= 1:
        score += 0.2
    return score


def conciseness_reward(text: str) -> float:
    """Reward concise but substantive responses."""
    wc = len(text.split())
    if wc < 10:
        return -1.0
    elif wc < 20:
        return -0.5
    elif wc <= 200:
        return 1.0
    elif wc <= 300:
        return 0.5
    return 0.0


def no_repetition_reward(text: str) -> float:
    """Penalise repetitive text."""
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if len(sentences) <= 1:
        return 0.0
    return len(set(sentences)) / len(sentences)


def helpfulness_reward(prompt: str, text: str) -> float:
    """Reward responses that address the prompt."""
    refusal_starts = [
        "i can't", "i cannot", "i'm sorry", "sorry,", "i apologize",
        "as an ai", "i'm not able",
    ]
    text_lower = text.lower().strip()
    if any(text_lower.startswith(r) for r in refusal_starts):
        return -0.5
    stop_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'what', 'how',
        'why', 'when', 'where', 'who', 'which', 'do', 'does', 'can',
        'could', 'would', 'should', 'in', 'on', 'at', 'to', 'for',
        'of', 'and', 'or', 'but', 'with', 'this', 'that', 'it', 'me',
        'you', 'your', 'my', 'i', 'please', 'explain', 'describe',
    }
    prompt_words = set(prompt.lower().split()) - stop_words
    response_words = set(text_lower.split())
    if prompt_words:
        overlap = len(prompt_words & response_words) / len(prompt_words)
    else:
        overlap = 0.5
    return min(overlap * 2.0, 1.0)


def score_response(prompt: str, text: str) -> dict:
    """Compute all reward scores for a single response."""
    return {
        "format": format_reward(text),
        "conciseness": conciseness_reward(text),
        "no_repetition": no_repetition_reward(text),
        "helpfulness": helpfulness_reward(prompt, text),
    }


print("Reward functions defined.")

## Model Registry

We define every model that should be evaluated. Each entry specifies whether it has a LoRA adapter and where to find it. Models are loaded one at a time.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

MODEL_REGISTRY = [
    {"name": "Base",              "adapter": None,                           "notebook": "-"},
    {"name": "SFT",               "adapter": "./results/sft/final",          "notebook": "03"},
    {"name": "RLHF-GRPO",         "adapter": "./results/ppo/final",          "notebook": "05"},
    {"name": "DPO",               "adapter": "./results/dpo/final",          "notebook": "06"},
    {"name": "GRPO-Custom",       "adapter": "./results/grpo/final",         "notebook": "08"},
    {"name": "f-GRPO (KL)",       "adapter": "./results/fgrpo_kl/final",     "notebook": "09"},
    {"name": "f-GRPO (RevKL)",    "adapter": "./results/fgrpo_reverse_kl/final", "notebook": "09"},
    {"name": "f-GRPO (Hellinger)","adapter": "./results/fgrpo_hellinger/final",   "notebook": "09"},
]

# Filter to models that actually exist on disk
available_models = []
for entry in MODEL_REGISTRY:
    if entry["adapter"] is None:
        available_models.append(entry)
        print(f"  [available]  {entry['name']:<22s} (base model)")
    elif os.path.isdir(entry["adapter"]):
        available_models.append(entry)
        print(f"  [available]  {entry['name']:<22s} ({entry['adapter']})")
    else:
        print(f"  [missing]    {entry['name']:<22s} ({entry['adapter']}) — skipping")

print(f"\nModels to evaluate: {len(available_models)} / {len(MODEL_REGISTRY)}")

## Generation Utilities

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_model(adapter_path=None):
    """Load the base model, optionally with a LoRA adapter."""
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model


def unload_model(model):
    """Free all GPU memory held by a model."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


@torch.no_grad()
def generate_response(model, prompt: str, max_new_tokens: int = 256) -> str:
    """Generate a single response using the chat template."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True)


print("Utilities ready.")

## Sequential Model Evaluation

We load each model one at a time, generate responses for all test prompts, compute reward scores, then free VRAM before moving to the next model. Results are cached in a dictionary.

In [ ]:
# Main evaluation loop
results = {}  # model_name -> {"responses": {prompt: text}, "scores": {prompt: {metric: val}}}

for entry in available_models:
    name = entry["name"]
    print(f"\n{'='*70}")
    print(f"Loading: {name}")
    print(f"{'='*70}")

    model = load_model(entry["adapter"])
    print(f"  VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

    responses = {}
    scores = {}

    for i, prompt in enumerate(all_prompts):
        resp = generate_response(model, prompt)
        responses[prompt] = resp
        scores[prompt] = score_response(prompt, resp)

        if (i + 1) % 5 == 0 or i == 0:
            print(f"  [{i+1:2d}/{len(all_prompts)}] {prompt[:50]}...")

    results[name] = {"responses": responses, "scores": scores}

    unload_model(model)
    print(f"  Unloaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

print(f"\n\nEvaluation complete for {len(results)} models.")

In [ ]:
# Save results to JSON for reproducibility
serialisable = {}
for model_name, data in results.items():
    serialisable[model_name] = {
        "responses": data["responses"],
        "scores": data["scores"],
    }

with open("./results/final_evaluation.json", "w") as f:
    json.dump(serialisable, f, indent=2)

print("Results saved to ./results/final_evaluation.json")

## Aggregate Reward Scores

Mean reward scores across all test prompts, broken down by reward function and overall.

In [ ]:
metrics = ["format", "conciseness", "no_repetition", "helpfulness"]

# Build summary dataframe
rows = []
for model_name, data in results.items():
    row = {"Model": model_name}
    for m in metrics:
        vals = [data["scores"][p][m] for p in all_prompts]
        row[m.title()] = np.mean(vals)
    row["Total"] = sum(row[m.title()] for m in metrics)
    rows.append(row)

df_summary = pd.DataFrame(rows).sort_values("Total", ascending=False).reset_index(drop=True)
df_summary.index = df_summary.index + 1  # 1-indexed rank
df_summary.index.name = "Rank"

print("=== Aggregate Reward Scores (mean across all test prompts) ===")
print()
print(df_summary.to_string(float_format="{:.3f}".format))

In [ ]:
# Grouped bar chart of reward scores
fig, ax = plt.subplots(figsize=(14, 6))

model_names = df_summary["Model"].tolist()
n_models = len(model_names)
n_metrics = len(metrics)
x = np.arange(n_models)
width = 0.8 / n_metrics
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

for i, (m, color) in enumerate(zip(metrics, colors)):
    vals = [df_summary.loc[df_summary["Model"] == name, m.title()].values[0] for name in model_names]
    ax.bar(x + i * width - (n_metrics - 1) * width / 2, vals, width,
           label=m.replace("_", " ").title(), color=color, edgecolor="black", linewidth=0.3)

ax.set_xlabel("Model (ranked by total score)")
ax.set_ylabel("Mean Reward Score")
ax.set_title("Reward Scores by Model and Metric")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=30, ha="right")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Radar Chart

A radar (spider) plot gives an at-a-glance profile of each model's strengths and weaknesses.

In [ ]:
radar_metrics = [m.title() for m in metrics]
n_metrics_r = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics_r, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

model_colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

for (model_name, data), color in zip(results.items(), model_colors):
    values = []
    for m in metrics:
        vals = [data["scores"][p][m] for p in all_prompts]
        values.append(np.mean(vals))
    values += values[:1]  # Close polygon
    ax.plot(angles, values, "o-", linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace("_", " ").title() for m in metrics], fontsize=11)
ax.set_title("Model Reward Profiles", fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Scores by Category

Do certain models excel on specific prompt types (factual, reasoning, instruction, creative, advice)?

In [ ]:
categories = list(TEST_PROMPTS.keys())

# Compute total reward per model per category
cat_scores = {}
for model_name, data in results.items():
    cat_scores[model_name] = {}
    for cat in categories:
        cat_prompts = TEST_PROMPTS[cat]
        totals = []
        for p in cat_prompts:
            s = data["scores"][p]
            totals.append(sum(s.values()))
        cat_scores[model_name][cat] = np.mean(totals)

# Heatmap
model_names_ordered = df_summary["Model"].tolist()
heat_data = np.array([[cat_scores[m][c] for c in categories] for m in model_names_ordered])

fig, ax = plt.subplots(figsize=(10, max(4, len(model_names_ordered) * 0.6)))
im = ax.imshow(heat_data, cmap="RdYlGn", aspect="auto")

ax.set_xticks(range(len(categories)))
ax.set_xticklabels([c.title() for c in categories], fontsize=11)
ax.set_yticks(range(len(model_names_ordered)))
ax.set_yticklabels(model_names_ordered, fontsize=10)

# Annotate cells
for i in range(len(model_names_ordered)):
    for j in range(len(categories)):
        ax.text(j, i, f"{heat_data[i, j]:.2f}", ha="center", va="center",
                fontsize=10, fontweight="bold",
                color="white" if heat_data[i, j] < 1.5 else "black")

ax.set_title("Total Reward Score by Model and Prompt Category", fontsize=13)
fig.colorbar(im, ax=ax, shrink=0.8, label="Total Reward")

plt.tight_layout()
plt.show()

## Response Length Analysis

Alignment training can change response length in systematic ways. Some methods (DPO, GRPO) may produce more concise outputs, while SFT can lead to verbosity.

In [ ]:
# Collect word counts per model
length_data = {}
for model_name, data in results.items():
    lengths = [len(data["responses"][p].split()) for p in all_prompts]
    length_data[model_name] = lengths

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
box_data = [length_data[m] for m in model_names_ordered]
bp = axes[0].boxplot(box_data, labels=model_names_ordered, patch_artist=True, vert=True)
for patch, color in zip(bp["boxes"], plt.cm.tab10(np.linspace(0, 1, len(model_names_ordered)))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel("Word Count")
axes[0].set_title("Response Length Distribution")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(True, alpha=0.3, axis="y")

# Mean length + std
means = [np.mean(length_data[m]) for m in model_names_ordered]
stds = [np.std(length_data[m]) for m in model_names_ordered]
bar_colors = plt.cm.tab10(np.linspace(0, 1, len(model_names_ordered)))

axes[1].barh(range(len(model_names_ordered)), means, xerr=stds,
             color=bar_colors, edgecolor="black", linewidth=0.3, capsize=3)
axes[1].set_yticks(range(len(model_names_ordered)))
axes[1].set_yticklabels(model_names_ordered)
axes[1].set_xlabel("Mean Word Count")
axes[1].set_title("Mean Response Length (with std)")
axes[1].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

# Summary table
print(f"{'Model':<22s} {'Mean':>6s} {'Median':>7s} {'Std':>6s} {'Min':>5s} {'Max':>5s}")
print("-" * 55)
for m in model_names_ordered:
    L = length_data[m]
    print(f"{m:<22s} {np.mean(L):>6.0f} {np.median(L):>7.0f} {np.std(L):>6.0f} {min(L):>5d} {max(L):>5d}")

## Vocabulary Diversity

We measure the **type-token ratio** (unique words / total words) as a simple proxy for vocabulary richness.

In [ ]:
diversity_data = {}
for model_name, data in results.items():
    all_words = []
    ttrs = []
    for p in all_prompts:
        words = data["responses"][p].lower().split()
        all_words.extend(words)
        if words:
            ttrs.append(len(set(words)) / len(words))
    diversity_data[model_name] = {
        "mean_ttr": np.mean(ttrs),
        "total_unique": len(set(all_words)),
        "total_words": len(all_words),
        "global_ttr": len(set(all_words)) / len(all_words) if all_words else 0,
    }

print(f"{'Model':<22s} {'Mean TTR':>9s} {'Global TTR':>11s} {'Unique Words':>13s} {'Total Words':>12s}")
print("-" * 70)
for m in model_names_ordered:
    d = diversity_data[m]
    print(f"{m:<22s} {d['mean_ttr']:>9.3f} {d['global_ttr']:>11.3f} {d['total_unique']:>13d} {d['total_words']:>12d}")

## Head-to-Head Win Rate Matrix

For each pair of models, we count how often one model achieves a higher total reward than the other on the same prompt. This gives a pairwise win-rate matrix.

In [ ]:
model_list = list(results.keys())
n = len(model_list)
win_matrix = np.zeros((n, n))

for i, m1 in enumerate(model_list):
    for j, m2 in enumerate(model_list):
        if i == j:
            win_matrix[i, j] = 0.5  # Self = draw
            continue
        wins = 0
        for p in all_prompts:
            s1 = sum(results[m1]["scores"][p].values())
            s2 = sum(results[m2]["scores"][p].values())
            if s1 > s2:
                wins += 1
            elif s1 == s2:
                wins += 0.5
        win_matrix[i, j] = wins / len(all_prompts)

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(win_matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(n))
ax.set_xticklabels(model_list, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(n))
ax.set_yticklabels(model_list, fontsize=9)
ax.set_xlabel("Opponent")
ax.set_ylabel("Model")
ax.set_title("Head-to-Head Win Rate (row model vs column model)", fontsize=12)

for i in range(n):
    for j in range(n):
        val = win_matrix[i, j]
        color = "white" if val < 0.35 or val > 0.65 else "black"
        ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                fontsize=9, fontweight="bold", color=color)

fig.colorbar(im, ax=ax, shrink=0.7, label="Win Rate")
plt.tight_layout()
plt.show()

# Overall win rates
print("\nOverall Win Rates (mean across all opponents):")
overall_wr = []
for i, m in enumerate(model_list):
    wr = np.mean([win_matrix[i, j] for j in range(n) if j != i])
    overall_wr.append((m, wr))

overall_wr.sort(key=lambda x: x[1], reverse=True)
for rank, (m, wr) in enumerate(overall_wr, 1):
    print(f"  {rank}. {m:<22s} {wr:.1%}")

## ELO Ratings

We compute ELO ratings from the pairwise comparisons. Each prompt is treated as a separate game between every pair of models.

In [ ]:
def compute_elo(results_dict, prompts, k=32, initial=1500, n_rounds=10):
    """Compute ELO ratings from pairwise reward comparisons.

    We run multiple rounds with shuffled prompt order for stability.
    """
    model_list = list(results_dict.keys())
    elo_accum = defaultdict(list)

    for _ in range(n_rounds):
        elo = {m: initial for m in model_list}
        shuffled = np.random.permutation(len(prompts))

        for idx in shuffled:
            p = prompts[idx]
            for i, m1 in enumerate(model_list):
                for m2 in model_list[i + 1:]:
                    s1 = sum(results_dict[m1]["scores"][p].values())
                    s2 = sum(results_dict[m2]["scores"][p].values())

                    # Expected scores
                    e1 = 1 / (1 + 10 ** ((elo[m2] - elo[m1]) / 400))
                    e2 = 1 - e1

                    # Actual scores
                    if s1 > s2:
                        a1, a2 = 1.0, 0.0
                    elif s2 > s1:
                        a1, a2 = 0.0, 1.0
                    else:
                        a1, a2 = 0.5, 0.5

                    elo[m1] += k * (a1 - e1)
                    elo[m2] += k * (a2 - e2)

        for m in model_list:
            elo_accum[m].append(elo[m])

    return {m: (np.mean(v), np.std(v)) for m, v in elo_accum.items()}


elo_ratings = compute_elo(results, all_prompts)

# Sort by rating
elo_sorted = sorted(elo_ratings.items(), key=lambda x: x[1][0], reverse=True)

print("=== ELO Ratings ===")
print(f"{'Rank':<5s} {'Model':<22s} {'ELO':>8s} {'Std':>6s}")
print("-" * 45)
for rank, (m, (rating, std)) in enumerate(elo_sorted, 1):
    print(f"{rank:<5d} {m:<22s} {rating:>8.0f} {std:>6.1f}")

# Plot
fig, ax = plt.subplots(figsize=(10, max(4, len(elo_sorted) * 0.5)))
names = [x[0] for x in elo_sorted]
ratings = [x[1][0] for x in elo_sorted]
stds = [x[1][1] for x in elo_sorted]
bar_colors = ["#FFD700" if i == 0 else "#C0C0C0" if i == 1 else "#CD7F32" if i == 2
              else "#4C72B0" for i in range(len(names))]

ax.barh(range(len(names)), ratings, xerr=stds, color=bar_colors,
        edgecolor="black", linewidth=0.5, capsize=3)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel("ELO Rating")
ax.set_title("Model Rankings by ELO")
ax.axvline(x=1500, color="gray", linestyle="--", linewidth=0.8, label="Initial (1500)")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Qualitative Comparison

Side-by-side responses on selected prompts to illustrate how alignment methods shape output style.

In [ ]:
showcase_prompts = [
    "List 5 tips for better sleep. Use a numbered list.",
    "What are the pros and cons of remote work?",
    "Write a motivational message for someone starting a new job.",
    "Explain how vaccines work.",
]

for prompt in showcase_prompts:
    print("\n" + "=" * 90)
    print(f"PROMPT: {prompt}")
    print("=" * 90)

    for model_name in results:
        resp = results[model_name]["responses"][prompt]
        s = results[model_name]["scores"][prompt]
        total = sum(s.values())
        wc = len(resp.split())

        print(f"\n--- {model_name} (reward={total:.2f}, words={wc}) ---")
        # Truncate long responses for readability
        display_text = resp[:400]
        if len(resp) > 400:
            display_text += "..."
        print(display_text)

## Method Comparison Summary

How do the alignment methods themselves compare?

In [ ]:
# Group models by method family
method_groups = {
    "Base (no alignment)": ["Base"],
    "SFT": ["SFT"],
    "RLHF / GRPO": ["RLHF-GRPO", "GRPO-Custom"],
    "DPO": ["DPO"],
    "f-GRPO": ["f-GRPO (KL)", "f-GRPO (RevKL)", "f-GRPO (Hellinger)"],
}

print(f"{'Method':<22s} {'Total Reward':>13s} {'Win Rate':>10s} {'ELO':>8s} {'Avg Words':>10s}")
print("-" * 67)

for method, model_names_in_group in method_groups.items():
    # Only include models we actually evaluated
    present = [m for m in model_names_in_group if m in results]
    if not present:
        continue

    # Average across all models in the group
    total_rewards = []
    win_rates = []
    elos = []
    word_counts = []

    for m in present:
        # Total reward
        tr = np.mean([sum(results[m]["scores"][p].values()) for p in all_prompts])
        total_rewards.append(tr)

        # Win rate
        idx = model_list.index(m)
        wr = np.mean([win_matrix[idx, j] for j in range(len(model_list)) if j != idx])
        win_rates.append(wr)

        # ELO
        if m in elo_ratings:
            elos.append(elo_ratings[m][0])

        # Word count
        wcs = [len(results[m]["responses"][p].split()) for p in all_prompts]
        word_counts.append(np.mean(wcs))

    print(f"{method:<22s} {np.mean(total_rewards):>13.3f} {np.mean(win_rates):>9.1%} "
          f"{np.mean(elos):>8.0f} {np.mean(word_counts):>10.0f}")

print("\n(Groups with multiple models show the mean across members)")

In [ ]:
# Final comprehensive ranking
print("\n" + "=" * 70)
print("FINAL MODEL RANKINGS")
print("=" * 70)
print()

# Combine multiple ranking signals
ranking_data = []
for m in results:
    tr = np.mean([sum(results[m]["scores"][p].values()) for p in all_prompts])
    idx = model_list.index(m)
    wr = np.mean([win_matrix[idx, j] for j in range(len(model_list)) if j != idx])
    elo = elo_ratings[m][0] if m in elo_ratings else 1500
    ranking_data.append({"Model": m, "Total Reward": tr, "Win Rate": wr, "ELO": elo})

df_rank = pd.DataFrame(ranking_data)

# Normalise each metric to [0, 1] and compute composite score
for col in ["Total Reward", "Win Rate", "ELO"]:
    lo, hi = df_rank[col].min(), df_rank[col].max()
    if hi > lo:
        df_rank[f"{col} (norm)"] = (df_rank[col] - lo) / (hi - lo)
    else:
        df_rank[f"{col} (norm)"] = 0.5

df_rank["Composite"] = (
    df_rank["Total Reward (norm)"] * 0.4
    + df_rank["Win Rate (norm)"] * 0.3
    + df_rank["ELO (norm)"] * 0.3
)

df_rank = df_rank.sort_values("Composite", ascending=False).reset_index(drop=True)
df_rank.index = df_rank.index + 1
df_rank.index.name = "Rank"

display_cols = ["Model", "Total Reward", "Win Rate", "ELO", "Composite"]
print(df_rank[display_cols].to_string(
    float_format="{:.3f}".format,
    formatters={"Win Rate": "{:.1%}".format, "ELO": "{:.0f}".format}
))

## Conclusion

### What We Evaluated

We compared up to 8 model variants produced across the alignment notebook series:

- **Base model** (Qwen2.5-7B-Instruct) — the pre-trained starting point
- **SFT** — supervised fine-tuning on human demonstrations
- **RLHF-GRPO** — reinforcement learning from a trained reward model
- **DPO** — direct preference optimisation on binary preference pairs
- **GRPO-Custom** — GRPO with hand-crafted reward functions
- **f-GRPO (KL, Reverse KL, Hellinger)** — divergence-based RL variants

### Key Findings

1. **All alignment methods improve over the base model** — even simple SFT produces measurable gains in structure and helpfulness.

2. **RL methods (GRPO, f-GRPO) are particularly effective at optimising specific reward signals** — they learn to format responses, control length, and address prompts more directly than SFT alone.

3. **DPO provides a strong balance** of quality and simplicity — no reward model needed, no generation during training.

4. **f-GRPO divergence choice matters** — different f-divergences produce models with different response profiles (length, diversity, reward distribution).

5. **The best method depends on your goals** — there is no single winner across all dimensions.

### The Alignment Pipeline

```
Pre-training → SFT → {RLHF / DPO / GRPO / f-GRPO} → Evaluation
```

Each stage builds on the previous one, and the choice of alignment method should be guided by:
- **Available data**: preference pairs → DPO; prompts only → GRPO/f-GRPO
- **Available compute**: DPO is cheapest; PPO most expensive; GRPO/f-GRPO in between
- **Desired properties**: if you need specific behaviours, rule-based GRPO gives the most control
- **Stability requirements**: f-GRPO with Hellinger/TV for maximum stability

### Complete Notebook Series

| # | Notebook | Method |
|---|----------|--------|
| 01 | Introduction to Alignment | Concepts and exploration |
| 02 | Exploring Alignment Datasets | Data preparation |
| 03 | Supervised Fine-Tuning | SFT with QLoRA |
| 04 | Reward Modeling | Train a reward model |
| 05 | RLHF with GRPO | RL from reward model |
| 06 | Direct Preference Optimization | DPO on preferences |
| 07 | Evaluation and Comparison | Mid-series evaluation |
| 08 | Group Relative Policy Optimization | GRPO with custom rewards |
| 09 | f-GRPO | Divergence-based RL |
| **10** | **Final Evaluation** | **Comprehensive comparison** |